# UEBA End-to-End Pipeline

Feature engineering (deviation scores) -> Point-wise Autoencoder -> LSTM
Sequence Autoencoder -> percentile threshold -> multi-model anomaly-type
classifier -> full end-to-end evaluation.

This notebook is NOT executed here. Run it yourself, top to bottom, after
placing `ueba_synthetic_dataset.csv` (the generator output, with `role`,
`geo_lat`, `geo_lon` columns already included) in the same folder.


## 1. Imports and config

In [ ]:
import json
import numpy as np
import pandas as pd
from math import radians, sin, cos, sqrt, atan2

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

CSV_PATH = "/content/ueba_synthetic_dataset_200k.csv"

# Time-based split: train on the earlier period, test on the later period.
# Adjust the cutoff date if your generator's START_DATE / NUM_DAYS differ.
TRAIN_END_DATE = pd.Timestamp("2025-09-29 00:00:00")

COMMAND_FREQ_THRESHOLD = 0.01      # min fraction of a role's sessions a command must appear in to be "expected"
COLD_START_MIN_EVENTS = 20         # min entity history before trusting entity-level over role-level baselines
WINDOW_SIZE = 5                    # LSTM sequence window length (events per window, per entity)

# =========================================================================
# >>> ALERT THRESHOLD PERCENTILE — CHANGE THIS NUMBER TO TUNE ALERT VOLUME <<<
# e.g. 0.99 = top 1% of events flagged as alerts, 0.95 = top 5%, etc.
ALERT_THRESHOLD_PERCENTILE = 0.98
# =========================================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cuda


## 2. Load data

In [ ]:
df = pd.read_csv("/content/ueba_synthetic_dataset_200k.csv")
df["timestamp"] = pd.to_datetime(df["timestamp"])
df["command_sequence"] = df["command_sequence"].apply(json.loads)
df = df.sort_values("timestamp").reset_index(drop=True)

print(f"Total rows: {len(df)}")
print(df["label"].value_counts())
df.head()


Total rows: 168681
label
normal                       164555
brute_force                     850
credential_stuffing             825
lateral_movement                770
insider_drift                   763
low_and_slow_exfiltration       497
device_spoofing                 257
impossible_travel               163
Name: count, dtype: int64


,entity_id,entity_type,role,timestamp,source_ip,geo_location,geo_lat,geo_lon,resource_accessed,auth_method,auth_result,session_duration_sec,command_sequence,device_fingerprint,mac_address,label,deviation_source
0,USR_185,user,Executive,2025-06-01 08:00:33,192.70.18.128,"Jillview, Montserrat",-12.048711,-115.999969,board-strategy-docs,password_mfa,success,1742,"[login, read_confidential_file, logout]",Windows 11 Executive,8a:85:f0:fb:2b:76,normal,none
1,SVC_015,service_account,service_account,2025-06-01 08:00:37,10.90.0.15,Internal DataCenter,-50.880434,102.317160,db-replica-sync,certificate,success,3446,"[authenticate, sync_database, terminate_session]",Linux Server,ae:65:0c:c8:bb:50,normal,none
2,SVC_014,service_account,service_account,2025-06-01 08:01:18,10.90.0.14,Internal DataCenter,-80.063370,-112.949997,backup-storage,certificate,success,2339,"[authenticate, sync_database, terminate_session]",Linux Server,8e:92:fd:32:07:62,normal,none
3,EDG_003,edge_device,edge_device,2025-06-01 08:01:55,192.168.51.104,"-9.4474465, 71.594498",-9.447446,71.594498,sensor-telemetry-endpoint,certificate,success,2247,"[connect, transmit_telemetry, disconnect]",fw_v2.3,4a:fc:75:63:8f:60,normal,none
4,EDG_014,edge_device,edge_device,2025-06-01 08:03:10,192.168.171.106,"23.627697, 59.363318",23.627697,59.363318,sensor-telemetry-endpoint,certificate,success,3310,"[connect, transmit_telemetry, disconnect]",fw_v2.3,6c:9e:35:20:d3:c5,normal,none


## 3. Train/test split (time-based)

In [ ]:
train_mask = df["timestamp"] < TRAIN_END_DATE
train_df = df[train_mask].copy()
test_df = df[~train_mask].copy()

train_normal_df = train_df[train_df["label"] == "normal"].copy()

print(f"Train rows: {len(train_df)} (normal-only: {len(train_normal_df)})")
print(f"Test rows: {len(test_df)}")


Train rows: 161042 (normal-only: 157120)
Test rows: 7639


## 4. Build reference baselines (from TRAIN NORMAL data only)

Plain dictionaries of stored statistics -- not models. Every deviation score
computed later looks these up. Frozen after this cell; never updated using
test-period or anomaly-labeled data.


In [ ]:
# --- 4a. Expected commands per role ---
def build_expected_commands(normal_df, freq_threshold=COMMAND_FREQ_THRESHOLD):
    expected_commands = {}
    for role, group in normal_df.groupby("role"):
        n_sessions = len(group)
        command_counts = {}
        for seq in group["command_sequence"]:
            for cmd in set(seq):
                command_counts[cmd] = command_counts.get(cmd, 0) + 1
        expected_commands[role] = {
            cmd for cmd, count in command_counts.items()
            if (count / n_sessions) >= freq_threshold
        }
    return expected_commands

expected_commands = build_expected_commands(train_normal_df)

# --- 4b. Known resources per entity (fallback: per role) ---
known_resources_entity = train_normal_df.groupby("entity_id")["resource_accessed"].apply(set).to_dict()
known_resources_role = train_normal_df.groupby("role")["resource_accessed"].apply(set).to_dict()

# --- 4c. Known devices/MACs per entity ---
known_devices_entity = (
    train_normal_df.groupby("entity_id")
    .apply(lambda g: set(zip(g["device_fingerprint"], g["mac_address"])))
    .to_dict()
)

# --- 4d. Session duration mean/std per entity (fallback: per role) ---
duration_stats_entity = train_normal_df.groupby("entity_id")["session_duration_sec"].agg(["mean", "std"]).to_dict("index")
duration_stats_role = train_normal_df.groupby("role")["session_duration_sec"].agg(["mean", "std"]).to_dict("index")

# --- 4e. Implied-speed mean/std per entity (fallback: per role), using geo_lat/geo_lon ---
def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    dlat = radians(lat2 - lat1)
    dlon = radians(lon2 - lon1)
    a = sin(dlat / 2) ** 2 + cos(radians(lat1)) * cos(radians(lat2)) * sin(dlon / 2) ** 2
    return 2 * R * atan2(sqrt(a), sqrt(1 - a))

def compute_implied_speed(sub_df):
    speeds = [np.nan]
    for i in range(1, len(sub_df)):
        prev = sub_df.iloc[i - 1]
        curr = sub_df.iloc[i]
        dist_km = haversine_km(prev["geo_lat"], prev["geo_lon"], curr["geo_lat"], curr["geo_lon"])
        time_gap_hr = max((curr["timestamp"] - prev["timestamp"]).total_seconds() / 3600.0, 1e-6)
        speeds.append(dist_km / time_gap_hr)
    return pd.Series(speeds, index=sub_df.index)

_train_normal_sorted = train_normal_df.sort_values(["entity_id", "timestamp"])
_speed_series_list = [compute_implied_speed(sub) for _, sub in _train_normal_sorted.groupby("entity_id")]
_train_normal_sorted = _train_normal_sorted.copy()
_train_normal_sorted["implied_speed_kmh"] = pd.concat(_speed_series_list).sort_index()

speed_stats_entity = _train_normal_sorted.groupby("entity_id")["implied_speed_kmh"].agg(["mean", "std"]).to_dict("index")
speed_stats_role = _train_normal_sorted.groupby("role")["implied_speed_kmh"].agg(["mean", "std"]).to_dict("index")

# --- 4f. Hour-of-day histogram per entity ---
_train_normal_hourly = train_normal_df.copy()
_train_normal_hourly["hour"] = _train_normal_hourly["timestamp"].dt.hour
hour_hist_entity = {
    entity_id: sub["hour"].value_counts(normalize=True).to_dict()
    for entity_id, sub in _train_normal_hourly.groupby("entity_id")
}

print("Reference baselines built.")


NameError: name 'train_normal_df' is not defined

## 5. Deviation-score functions

In [ ]:
def command_novelty_score(row):
    role = row["role"]
    cmds = set(row["command_sequence"])
    expected = expected_commands.get(role, set())
    if len(cmds) == 0:
        return 0.0
    return len(cmds - expected) / len(cmds)


def resource_novelty_score(row):
    entity_id = row["entity_id"]
    resource = row["resource_accessed"]
    ent_known = known_resources_entity.get(entity_id, set())
    if len(ent_known) >= COLD_START_MIN_EVENTS:
        known = ent_known
    else:
        known = known_resources_role.get(row["role"], set())
    return 0.0 if resource in known else 1.0


def device_novelty_score(row):
    entity_id = row["entity_id"]
    pair = (row["device_fingerprint"], row["mac_address"])
    known = known_devices_entity.get(entity_id, set())
    return 0.0 if pair in known else 1.0


def duration_zscore(row):
    stats = duration_stats_entity.get(row["entity_id"])
    if stats is None or pd.isna(stats.get("std")) or stats.get("std", 0) == 0:
        stats = duration_stats_role.get(row["role"], {"mean": row["session_duration_sec"], "std": 1.0})
    mean = stats["mean"]
    std = stats["std"] if stats["std"] not in (0, None) and not pd.isna(stats["std"]) else 1.0
    return (row["session_duration_sec"] - mean) / std


def time_unusualness_score(row):
    hist = hour_hist_entity.get(row["entity_id"], {})
    prob = hist.get(row["timestamp"].hour, 0.0)
    return -np.log(prob + 1e-3)


def hour_cyclical_features(row):
    angle = 2 * np.pi * row["timestamp"].hour / 24
    return np.sin(angle), np.cos(angle)


def geo_velocity_zscore(row, prev_row):
    """prev_row: this entity's immediately preceding row, or None if this is
    the entity's first event (returns 0.0 in that case -- no prior point to compare).
    """
    if prev_row is None:
        return 0.0
    dist_km = haversine_km(prev_row["geo_lat"], prev_row["geo_lon"], row["geo_lat"], row["geo_lon"])
    time_gap_hr = max((row["timestamp"] - prev_row["timestamp"]).total_seconds() / 3600.0, 1e-6)
    implied_speed = dist_km / time_gap_hr

    stats = speed_stats_entity.get(row["entity_id"])
    if stats is None or pd.isna(stats.get("std")) or stats.get("std", 0) == 0:
        stats = speed_stats_role.get(row["role"], {"mean": 0.0, "std": 1.0})
    mean = stats["mean"] if not pd.isna(stats.get("mean", np.nan)) else 0.0
    std = stats["std"] if stats.get("std") not in (0, None) and not pd.isna(stats.get("std", np.nan)) else 1.0
    return (implied_speed - mean) / std


## 6. Assemble full feature table

Applies every deviation-score function row-wise, per entity (so
geo_velocity_zscore can see each entity's own previous row). Produces one row
per event with all numeric deviation features plus role/entity_type one-hot
columns.


In [ ]:
def build_features_for_entity(entity_df):
    entity_df = entity_df.sort_values("timestamp").reset_index(drop=True)
    rows = []
    prev_row = None
    for _, row in entity_df.iterrows():
        rows.append({
            "entity_id": row["entity_id"],
            "timestamp": row["timestamp"],
            "role": row["role"],
            "entity_type": row["entity_type"],
            "label": row["label"],
            "command_novelty_score": command_novelty_score(row),
            "resource_novelty_score": resource_novelty_score(row),
            "device_novelty_score": device_novelty_score(row),
            "duration_zscore": duration_zscore(row),
            "time_unusualness_score": time_unusualness_score(row),
            "geo_velocity_zscore": geo_velocity_zscore(row, prev_row),
            "recent_failed_auth_count": 0,  # filled in Section 6a below
        })
        rows[-1]["hour_sin"], rows[-1]["hour_cos"] = hour_cyclical_features(row)
        prev_row = row
    return pd.DataFrame(rows)


feature_rows = [build_features_for_entity(sub) for _, sub in df.groupby("entity_id")]
feature_df = pd.concat(feature_rows, ignore_index=False)
feature_df = feature_df.set_index(df.sort_values("entity_id").index if False else feature_df.index)
# Re-align to df's original row order/index by merging on (entity_id, timestamp)
feature_df = feature_df.reset_index(drop=True)


KeyboardInterrupt: 

### 6a. Recent failed-auth rolling count (per entity, last 5 minutes)

In [ ]:
def compute_recent_failed_auth_counts(df, window_minutes=5):
    df_sorted = df.sort_values(["entity_id", "timestamp"]).copy()
    result = []
    for entity_id, sub in df_sorted.groupby("entity_id"):
        sub = sub.sort_values("timestamp")
        times = sub["timestamp"].values
        fails = (sub["auth_result"] == "fail").values
        window = pd.Timedelta(minutes=window_minutes).to_timedelta64()
        counts = np.zeros(len(sub), dtype=int)
        left = 0
        fail_in_window = 0
        for right in range(len(sub)):
            if fails[right]:
                fail_in_window += 1
            while times[right] - times[left] > window:
                if fails[left]:
                    fail_in_window -= 1
                left += 1
            counts[right] = fail_in_window
        result.append(pd.Series(counts, index=sub.index))
    df_sorted["recent_failed_auth_count"] = pd.concat(result).sort_index()
    return df_sorted

df_with_counts = compute_recent_failed_auth_counts(df)

# Merge the rolling failed-auth count into feature_df by (entity_id, timestamp)
feature_df = feature_df.merge(
    df_with_counts[["entity_id", "timestamp", "recent_failed_auth_count"]].rename(
        columns={"recent_failed_auth_count": "recent_failed_auth_count_actual"}
    ),
    on=["entity_id", "timestamp"], how="left",
)
feature_df["recent_failed_auth_count"] = np.log1p(feature_df["recent_failed_auth_count_actual"].fillna(0))
feature_df = feature_df.drop(columns=["recent_failed_auth_count_actual"])

feature_df = feature_df.sort_values(["entity_id", "timestamp"]).reset_index(drop=True)
feature_df.head()


In [ ]:
NUMERIC_COLS = [
    "command_novelty_score", "resource_novelty_score", "device_novelty_score",
    "duration_zscore", "time_unusualness_score", "geo_velocity_zscore",
    "hour_sin", "hour_cos", "recent_failed_auth_count",
]

role_dummies = pd.get_dummies(feature_df["role"], prefix="role")
etype_dummies = pd.get_dummies(feature_df["entity_type"], prefix="etype")

FEATURE_COLUMNS = NUMERIC_COLS + list(role_dummies.columns) + list(etype_dummies.columns)

full_feature_df = pd.concat(
    [feature_df[["entity_id", "timestamp", "label"] + NUMERIC_COLS].reset_index(drop=True),
     role_dummies.reset_index(drop=True),
     etype_dummies.reset_index(drop=True)],
    axis=1,
)

full_feature_df.head()


## 7. Split features, scale numeric columns

Scaler statistics (mean/std) are fit on TRAIN-NORMAL rows only, then applied
unchanged to train-all and test data (no leakage).


In [ ]:
train_feature_mask = full_feature_df["timestamp"] < TRAIN_END_DATE
train_features_all = full_feature_df[train_feature_mask].copy()
test_features_all = full_feature_df[~train_feature_mask].copy()
train_features_normal = train_features_all[train_features_all["label"] == "normal"].copy()

feature_means = train_features_normal[NUMERIC_COLS].mean()
feature_stds = train_features_normal[NUMERIC_COLS].std().replace(0, 1.0)

def scale_numeric(d):
    d = d.copy()
    d[NUMERIC_COLS] = (d[NUMERIC_COLS] - feature_means) / feature_stds
    return d

train_features_normal_scaled = scale_numeric(train_features_normal)
train_features_all_scaled = scale_numeric(train_features_all)
test_features_all_scaled = scale_numeric(test_features_all)

X_train_normal = train_features_normal_scaled[FEATURE_COLUMNS].astype(float).values
X_train_all = train_features_all_scaled[FEATURE_COLUMNS].astype(float).values
X_test_all = test_features_all_scaled[FEATURE_COLUMNS].astype(float).values

print("X_train_normal:", X_train_normal.shape)
print("X_train_all:", X_train_all.shape)
print("X_test_all:", X_test_all.shape)


## 8. Point-wise Autoencoder — model, training

In [ ]:
class PointDataset(Dataset):
    def __init__(self, X):
        self.X = torch.tensor(X, dtype=torch.float32)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx]


class Autoencoder(nn.Module):
    def __init__(self, input_dim, latent_dim=8):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 32), nn.ReLU(),
            nn.Linear(32, 16), nn.ReLU(),
            nn.Linear(16, latent_dim),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 16), nn.ReLU(),
            nn.Linear(16, 32), nn.ReLU(),
            nn.Linear(32, input_dim),
        )
    def forward(self, x):
        return self.decoder(self.encoder(x))


POINT_LATENT_DIM = 8
POINT_BATCH_SIZE = 64
POINT_NUM_EPOCHS = 50
POINT_LR = 1e-3

point_train_loader = DataLoader(PointDataset(X_train_normal), batch_size=POINT_BATCH_SIZE, shuffle=True)

point_model = Autoencoder(input_dim=X_train_normal.shape[1], latent_dim=POINT_LATENT_DIM).to(device)
point_optimizer = torch.optim.Adam(point_model.parameters(), lr=POINT_LR)
point_criterion = nn.MSELoss()

point_model.train()
for epoch in range(POINT_NUM_EPOCHS):
    epoch_loss = 0.0
    for batch in point_train_loader:
        batch = batch.to(device)
        point_optimizer.zero_grad()
        recon = point_model(batch)
        loss = point_criterion(recon, batch)
        loss.backward()
        point_optimizer.step()
        epoch_loss += loss.item() * batch.size(0)
    epoch_loss /= len(point_train_loader.dataset)
    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f"[Point AE] Epoch {epoch + 1}/{POINT_NUM_EPOCHS} - MSE: {epoch_loss:.5f}")


## 9. LSTM Sequence Autoencoder — windows, model, training

In [ ]:
def build_windows_for_entity(entity_df, window_size, feature_columns):
    windows = []
    values = entity_df[feature_columns].astype(float).values
    labels = entity_df["label"].values
    n = len(entity_df)
    for end in range(window_size - 1, n):
        start = end - window_size + 1
        windows.append((values[start:end + 1], labels[start:end + 1], entity_df.index[end]))
    return windows

def build_all_windows(d, window_size, feature_columns):
    all_windows = []
    for entity_id, sub in d.groupby("entity_id"):
        sub = sub.sort_values("timestamp")
        all_windows.extend(build_windows_for_entity(sub, window_size, feature_columns))
    return all_windows

train_windows_all = build_all_windows(train_features_all_scaled, WINDOW_SIZE, FEATURE_COLUMNS)
test_windows_all = build_all_windows(test_features_all_scaled, WINDOW_SIZE, FEATURE_COLUMNS)
train_windows_normal = [w for w in train_windows_all if all(l == "normal" for l in w[1])]

print(f"Train windows (normal-only): {len(train_windows_normal)}")
print(f"Test windows: {len(test_windows_all)}")


In [ ]:
class WindowDataset(Dataset):
    def __init__(self, windows):
        self.X = torch.tensor(np.stack([w[0] for w in windows]), dtype=torch.float32)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx]


class LSTMAutoencoder(nn.Module):
    def __init__(self, input_dim, hidden_dim=16, latent_dim=8):
        super().__init__()
        self.encoder_lstm = nn.LSTM(input_dim, hidden_dim, batch_first=True)
        self.to_latent = nn.Linear(hidden_dim, latent_dim)
        self.from_latent = nn.Linear(latent_dim, hidden_dim)
        self.decoder_lstm = nn.LSTM(input_dim, hidden_dim, batch_first=True)
        self.output_layer = nn.Linear(hidden_dim, input_dim)

    def forward(self, x, teacher_forcing=True):
        batch_size, window_size, input_dim = x.shape
        _, (h_n, c_n) = self.encoder_lstm(x)
        h_n = h_n.squeeze(0)
        z = self.to_latent(h_n)

        decoder_hidden = self.from_latent(z).unsqueeze(0)
        decoder_cell = torch.zeros_like(decoder_hidden)
        outputs = []
        decoder_input = torch.zeros(batch_size, 1, input_dim, device=x.device)

        for t in range(window_size):
            out, (decoder_hidden, decoder_cell) = self.decoder_lstm(decoder_input, (decoder_hidden, decoder_cell))
            step_output = self.output_layer(out)
            outputs.append(step_output)
            decoder_input = x[:, t:t + 1, :] if teacher_forcing else step_output

        return torch.cat(outputs, dim=1)


LSTM_HIDDEN_DIM = 16
LSTM_LATENT_DIM = 8
LSTM_NUM_EPOCHS = 50
LSTM_LR = 1e-3

lstm_train_loader = DataLoader(WindowDataset(train_windows_normal), batch_size=64, shuffle=True)

lstm_model = LSTMAutoencoder(input_dim=len(FEATURE_COLUMNS), hidden_dim=LSTM_HIDDEN_DIM, latent_dim=LSTM_LATENT_DIM).to(device)
lstm_optimizer = torch.optim.Adam(lstm_model.parameters(), lr=LSTM_LR)

lstm_model.train()
for epoch in range(LSTM_NUM_EPOCHS):
    epoch_loss = 0.0
    for batch in lstm_train_loader:
        batch = batch.to(device)
        lstm_optimizer.zero_grad()
        recon = lstm_model(batch, teacher_forcing=True)
        loss = ((recon - batch) ** 2).mean()
        loss.backward()
        lstm_optimizer.step()
        epoch_loss += loss.item() * batch.size(0)
    epoch_loss /= len(lstm_train_loader.dataset)
    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f"[LSTM AE] Epoch {epoch + 1}/{LSTM_NUM_EPOCHS} - MSE: {epoch_loss:.5f}")


## 10. Scoring functions (point-wise MSE, LSTM max-over-timesteps)

In [ ]:
def score_point(model, X, device):
    model.eval()
    with torch.no_grad():
        X_t = torch.tensor(X, dtype=torch.float32).to(device)
        recon = model(X_t)
        errors = torch.mean((recon - X_t) ** 2, dim=1)
    return errors.cpu().numpy()


def score_windows(model, windows, device, batch_size=256):
    model.eval()
    results = []
    with torch.no_grad():
        for i in range(0, len(windows), batch_size):
            batch_windows = windows[i:i + batch_size]
            X = torch.tensor(np.stack([w[0] for w in batch_windows]), dtype=torch.float32).to(device)
            recon = model(X, teacher_forcing=False)
            per_step_error = ((recon - X) ** 2).mean(dim=2)  # (batch, window_size)
            max_error = per_step_error.max(dim=1).values.cpu().numpy()
            for j, w in enumerate(batch_windows):
                results.append({"row_index": w[2], "label": w[1][-1], "seq_max_error": max_error[j]})
    return pd.DataFrame(results)


In [ ]:
# Point-wise scores
train_point_errors_all = score_point(point_model, X_train_all, device)
test_point_errors_all = score_point(point_model, X_test_all, device)

train_features_all_scaled = train_features_all_scaled.reset_index(drop=True)
test_features_all_scaled = test_features_all_scaled.reset_index(drop=True)

train_point_scores = train_features_all_scaled[["entity_id", "timestamp", "label"]].copy()
train_point_scores["point_error"] = train_point_errors_all

test_point_scores = test_features_all_scaled[["entity_id", "timestamp", "label"]].copy()
test_point_scores["point_error"] = test_point_errors_all

# Sequence scores
train_seq_scores = score_windows(lstm_model, train_windows_all, device)
test_seq_scores = score_windows(lstm_model, test_windows_all, device)


## 11. Combine point-wise + sequence scores into one final risk score

Percentile-normalize both independently, then take the max (either model
alone is sufficient to raise risk). Events too early in an entity's history to
have a full window (cold-start on window length) fall back to the point-wise
score alone.


In [ ]:
def combine_scores(point_scores_df, seq_scores_df, features_scaled_df):
    features_scaled_df = features_scaled_df.reset_index().rename(columns={"index": "row_index"})
    combined = point_scores_df.reset_index().rename(columns={"index": "row_index"})
    combined["point_score_pct"] = combined["point_error"].rank(pct=True)

    seq_scores_df = seq_scores_df.copy()
    seq_scores_df["seq_score_pct"] = seq_scores_df["seq_max_error"].rank(pct=True)

    combined = combined.merge(
        seq_scores_df[["row_index", "seq_score_pct"]], on="row_index", how="left"
    )
    combined["seq_score_pct"] = combined["seq_score_pct"].fillna(combined["point_score_pct"])
    combined["final_risk_score"] = combined[["point_score_pct", "seq_score_pct"]].max(axis=1)
    return combined

train_combined = combine_scores(train_point_scores, train_seq_scores, train_features_all_scaled)
test_combined = combine_scores(test_point_scores, test_seq_scores, test_features_all_scaled)

test_combined.head()


## 12. Alert threshold (percentile-based)

Threshold is computed on the TEST period's score distribution, at the
percentile set in Section 1 (`ALERT_THRESHOLD_PERCENTILE`). Change that single
variable back in Section 1 to make alerting stricter (closer to 1.0, fewer
alerts) or looser (lower, more alerts) -- everything below re-runs
automatically off that one number.


In [ ]:
threshold = test_combined["final_risk_score"].quantile(98)
test_combined["flagged"] = test_combined["final_risk_score"] >= threshold

print(f"Alert threshold percentile: {ALERT_THRESHOLD_PERCENTILE}")
print(f"Threshold value: {threshold:.4f}")
print(f"Rows flagged: {test_combined['flagged'].sum()} / {len(test_combined)}")
print("\nFlagged rows by true label:")
print(test_combined[test_combined['flagged']]['label'].value_counts())
print("\nMissed anomalies (label != normal, not flagged):")
missed = test_combined[(~test_combined['flagged']) & (test_combined['label'] != 'normal')]
print(missed['label'].value_counts())


## 13. Stage-2 classifiers — train on the labeled ANOMALY data

Trains several classifiers to predict anomaly TYPE (brute_force,
impossible_travel, credential_stuffing, lateral_movement, device_spoofing,
low_and_slow_exfiltration, insider_drift), using the same deviation-score
feature vectors. Trained on all anomaly-labeled rows across train+test period
(more data for the classifier to learn each attack's shape) -- normal rows are
excluded since this stage only ever runs on already-flagged, presumed-anomalous
events.

Includes: Decision Tree, Random Forest, Neural Network (MLP), and Isolation
Forest. Isolation Forest is unsupervised/semi-supervised by design (it doesn't
natively predict a class label) -- handled separately, see 13d.


In [ ]:
anomaly_feature_df = full_feature_df[full_feature_df["label"] != "normal"].copy()
X_anomaly = anomaly_feature_df[FEATURE_COLUMNS].astype(float).values

# Scale using the SAME scaler fit on train-normal data (Section 7) for consistency
# with what the autoencoders and inference pipeline use.
X_anomaly_scaled = anomaly_feature_df.copy()
X_anomaly_scaled[NUMERIC_COLS] = (X_anomaly_scaled[NUMERIC_COLS] - feature_means) / feature_stds
X_anomaly_final = X_anomaly_scaled[FEATURE_COLUMNS].astype(float).values

label_encoder = LabelEncoder()
y_anomaly = label_encoder.fit_transform(anomaly_feature_df["label"])

print("Anomaly-type class distribution:")
print(anomaly_feature_df["label"].value_counts())
print("\nEncoded classes:", list(label_encoder.classes_))


### 13a. Train/test split for the classifier (separate from the autoencoder's time-based split — stratified, since class counts are small)

In [ ]:
from sklearn.model_selection import train_test_split

X_clf_train, X_clf_test, y_clf_train, y_clf_test = train_test_split(
    X_anomaly_final, y_anomaly, test_size=0.3, random_state=RANDOM_SEED, stratify=y_anomaly
)

print(f"Classifier train rows: {len(X_clf_train)}, test rows: {len(X_clf_test)}")


### 13b. Train Decision Tree, Random Forest, Neural Network (MLP)

In [ ]:
decision_tree = DecisionTreeClassifier(random_state=RANDOM_SEED, class_weight="balanced")
decision_tree.fit(X_clf_train, y_clf_train)

random_forest = RandomForestClassifier(
    n_estimators=200, random_state=RANDOM_SEED, class_weight="balanced"
)
random_forest.fit(X_clf_train, y_clf_train)

neural_net = MLPClassifier(
    hidden_layer_sizes=(32, 16), max_iter=1000, random_state=RANDOM_SEED
)
neural_net.fit(X_clf_train, y_clf_train)

print("Decision Tree, Random Forest, Neural Network trained.")


### 13c. Isolation Forest

Isolation Forest is an anomaly-scoring model, not a multi-class classifier — it
outputs "anomalous vs. not," not a specific attack type. We include it here as
an ADDITIONAL binary check (does it agree this is anomalous at all) rather than
a type-predictor. Trained on train-normal data (same convention as the
autoencoders: it learns what "normal" looks like, then scores deviation from
that) — NOT on the anomaly-only set the other three classifiers use.


In [ ]:
isolation_forest = IsolationForest(
    n_estimators=200, contamination="auto", random_state=RANDOM_SEED
)
isolation_forest.fit(X_train_normal)  # trained on normal data, same convention as the autoencoders

print("Isolation Forest trained.")


### 13d. Classifier evaluation on their own held-out split

In [ ]:
def evaluate_classifier(name, model, X_test, y_test, label_encoder):
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average="weighted", zero_division=0)
    rec = recall_score(y_test, y_pred, average="weighted", zero_division=0)
    f1 = f1_score(y_test, y_pred, average="weighted", zero_division=0)

    print(f"\n=== {name} ===")
    print(f"Accuracy:  {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall:    {rec:.4f}")
    print(f"F1:        {f1:.4f}")
    print(classification_report(y_test, y_pred, target_names=label_encoder.classes_, zero_division=0))
    return {"model": name, "accuracy": acc, "precision": prec, "recall": rec, "f1": f1}


classifier_results = []
classifier_results.append(evaluate_classifier("Decision Tree", decision_tree, X_clf_test, y_clf_test, label_encoder))
classifier_results.append(evaluate_classifier("Random Forest", random_forest, X_clf_test, y_clf_test, label_encoder))
classifier_results.append(evaluate_classifier("Neural Network (MLP)", neural_net, X_clf_test, y_clf_test, label_encoder))

pd.DataFrame(classifier_results)


## 14. Full end-to-end pipeline test

For every TEST-period row: if its combined risk score cleared the alert
threshold (Section 12), pass it through each trained classifier to predict the
anomaly type. Then compare each classifier's predictions against the true
labels, but ONLY on the rows that were actually flagged — this measures the
real production pipeline (detection stage -> classification stage) rather than
the classifier in isolation.


In [ ]:
flagged_rows = test_combined[test_combined["flagged"]].copy()
flagged_row_indices = flagged_rows["row_index"].values

flagged_features = test_features_all_scaled.loc[flagged_row_indices]
X_flagged = flagged_features[FEATURE_COLUMNS].astype(float).values
y_flagged_true_raw = flagged_features["label"].values  # includes "normal" -- false positives from stage 1

print(f"Flagged rows going into Stage 2: {len(flagged_rows)}")
print(pd.Series(y_flagged_true_raw).value_counts())


### 14a. Restrict end-to-end scoring to flagged rows whose true label is an anomaly type

Rows flagged by Stage 1 but whose TRUE label is "normal" are Stage-1 false
positives — the anomaly-type classifiers were never trained on a "normal"
class, so they cannot meaningfully be scored on those rows. Report the Stage-1
false-positive count separately (Section 12 already shows this), and evaluate
Stage-2 classification accuracy only on flagged rows that are true anomalies.


In [ ]:
is_true_anomaly = y_flagged_true_raw != "normal"
X_flagged_anomalies = X_flagged[is_true_anomaly]
y_flagged_anomalies_raw = y_flagged_true_raw[is_true_anomaly]

print(f"Flagged rows that are true anomalies (evaluated below): {len(X_flagged_anomalies)}")
print(f"Flagged rows that are Stage-1 false positives (excluded from Stage-2 scoring): {(~is_true_anomaly).sum()}")

# Some anomaly types in this flagged subset may not appear in every class the
# classifier was trained on -- filter to only labels the encoder knows about.
known_label_mask = np.isin(y_flagged_anomalies_raw, label_encoder.classes_)
X_flagged_eval = X_flagged_anomalies[known_label_mask]
y_flagged_eval_raw = y_flagged_anomalies_raw[known_label_mask]
y_flagged_eval = label_encoder.transform(y_flagged_eval_raw)

print(f"Rows with a known label for evaluation: {len(X_flagged_eval)}")


### 14b. Run each classifier on the flagged+true-anomaly rows, report full metrics

In [ ]:
def evaluate_end_to_end(name, model, X, y_true, label_encoder):
    if len(X) == 0:
        print(f"\n=== {name} (END-TO-END) === No flagged true-anomaly rows to evaluate.")
        return {"model": name, "accuracy": None, "precision": None, "recall": None, "f1": None}

    y_pred = model.predict(X)
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average="weighted", zero_division=0)
    rec = recall_score(y_true, y_pred, average="weighted", zero_division=0)
    f1 = f1_score(y_true, y_pred, average="weighted", zero_division=0)

    print(f"\n=== {name} (END-TO-END: Stage1-flagged -> Stage2-classified) ===")
    print(f"Accuracy:  {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall:    {rec:.4f}")
    print(f"F1:        {f1:.4f}")
    labels_present = np.unique(np.concatenate([y_true, y_pred]))
    print(classification_report(
        y_true, y_pred,
        labels=labels_present,
        target_names=[label_encoder.classes_[i] for i in labels_present],
        zero_division=0,
    ))
    print("Confusion matrix (rows=true, cols=predicted):")
    print(confusion_matrix(y_true, y_pred, labels=labels_present))
    return {"model": name, "accuracy": acc, "precision": prec, "recall": rec, "f1": f1}


end_to_end_results = []
end_to_end_results.append(evaluate_end_to_end("Decision Tree", decision_tree, X_flagged_eval, y_flagged_eval, label_encoder))
end_to_end_results.append(evaluate_end_to_end("Random Forest", random_forest, X_flagged_eval, y_flagged_eval, label_encoder))
end_to_end_results.append(evaluate_end_to_end("Neural Network (MLP)", neural_net, X_flagged_eval, y_flagged_eval, label_encoder))

print("\n\n=== Summary: end-to-end (Stage1 detection -> Stage2 classification) ===")
pd.DataFrame(end_to_end_results)


### 14c. Isolation Forest — binary agreement check on flagged rows

Reports what fraction of Stage-1-flagged rows Isolation Forest (trained
independently, on normal data) ALSO considers anomalous — a cross-check
between two independently-trained detectors, not a type classification.


In [ ]:
iso_predictions = isolation_forest.predict(X_flagged)  # 1 = normal, -1 = anomaly, per sklearn convention
iso_agrees_anomalous = (iso_predictions == -1)

print(f"Of {len(X_flagged)} Stage-1-flagged rows, Isolation Forest also flags "
      f"{iso_agrees_anomalous.sum()} ({iso_agrees_anomalous.mean()*100:.1f}%) as anomalous.")


## 15. Stage-1 detection-only metrics (point-wise + LSTM combined, before classification)

Precision/recall of the DETECTION stage itself (flagged vs. not, collapsing all
anomaly types into one "anomalous" class) — separate from the Stage-2
type-classification metrics above. This answers "how good is the detector,"
independent of "how good is the classifier."


In [ ]:
y_detection_true = (test_combined["label"] != "normal").astype(int)
y_detection_pred = test_combined["flagged"].astype(int)

print("=== Stage 1 detection-only metrics (binary: anomalous vs normal) ===")
print(f"Accuracy:  {accuracy_score(y_detection_true, y_detection_pred):.4f}")
print(f"Precision: {precision_score(y_detection_true, y_detection_pred, zero_division=0):.4f}")
print(f"Recall:    {recall_score(y_detection_true, y_detection_pred, zero_division=0):.4f}")
print(f"F1:        {f1_score(y_detection_true, y_detection_pred, zero_division=0):.4f}")
print("\nConfusion matrix (rows=true[normal,anomalous], cols=pred[normal,anomalous]):")
print(confusion_matrix(y_detection_true, y_detection_pred))


## Notes / known limitations to state in your report

- `ALERT_THRESHOLD_PERCENTILE` (Section 1) controls the whole pipeline's alert
  volume — lower it to catch more anomalies at the cost of more false
  positives, raise it for the opposite tradeoff. Try a few values and report
  the precision/recall tradeoff curve.
- Stage-2 classifiers are trained on ALL anomaly-labeled rows (not just ones
  Stage 1 would have flagged) for more training data — but Section 14's
  end-to-end numbers are the realistic production metric, since only flagged
  rows ever reach Stage 2 in practice.
- Class imbalance among anomaly types (e.g. far fewer `impossible_travel` or
  `device_spoofing` rows than `credential_stuffing`) is handled via
  `class_weight="balanced"` for Decision Tree / Random Forest; MLPClassifier
  has no native class-weighting in sklearn, worth noting as a limitation or
  swapping in manual oversampling if MLP underperforms on rare classes.
- Isolation Forest is included as required, but framed correctly: it's a
  second independent anomaly detector, not a type-classifier — Section 14c
  reports agreement rate, not classification accuracy, for this reason.
